# Name That LLM · 词级 TF-IDF 基线

用训练集中的回答正文识别 18 个模型。运行全部单元，使用 CPU 即可。在 Kaggle 上请先为 Notebook 添加本比赛的数据。本地运行需要 Python、pandas、scikit-learn、joblib 和 pyarrow。

## 文本特征与分类器

将单词及相邻词对编码为 TF-IDF，用线性支持向量分类器预测模型名。词表和 IDF 仅在训练文本上拟合，再应用于当前测试集。

In [ ]:
"""Fit a text-only TF-IDF attribution model and generate a Kaggle submission."""

from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


def make_model(analyzer="word", ngram_range=(1, 2)):
    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        lowercase=True,
        min_df=2,
        sublinear_tf=True,
        norm="l2",
    )
    return Pipeline([
        ("tfidf", vectorizer),
        ("svc", LinearSVC(C=1.0, dual=True, random_state=42)),
    ])


def run(data_dir, output_path, model_out=None):
    data_dir = Path(data_dir)
    output_path = Path(output_path)
    train = pd.read_csv(data_dir / "train.csv", dtype=str, keep_default_na=False)
    test = pd.read_csv(data_dir / "test.csv", dtype=str, keep_default_na=False)
    for frame in [train, test]:
        assert frame["id"].is_unique and frame["id"].str.len().min() > 0
        assert frame["text"].str.strip().str.len().min() > 0
    assert train["model"].nunique() == 18
    model = make_model()
    model.fit(train["text"], train["model"])
    submission = pd.DataFrame({"id": test["id"], "model": model.predict(test["text"])})
    assert submission["model"].isin(train["model"].unique()).all()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_parquet(output_path, engine="pyarrow", index=False)
    saved = pd.read_parquet(output_path)
    assert saved.columns.tolist() == ["id", "model"]
    assert saved["id"].tolist() == test["id"].tolist()
    assert saved["model"].tolist() == submission["model"].tolist()
    if model_out is not None:
        Path(model_out).parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, model_out, compress=3)
    print(f"{len(train)} train rows, {len(saved)} predictions written to {output_path}")
    return saved

## 在全部训练数据上拟合并写入提交文件

逐条预测当前挂载的 `test.csv`。正式提交时 Kaggle 会用隐藏测试集重跑此 Notebook。输出按测试集顺序保留 `id`，并在 `model` 列写入预测模型名。

In [ ]:
kaggle_input = Path("/kaggle/input")
if kaggle_input.is_dir():
    candidates = [kaggle_input / "competitions" / "name-that-llm", kaggle_input / "name-that-llm"]
    data_dir = next((d for d in candidates if (d / "train.csv").is_file() and (d / "test.csv").is_file()), None)
    if data_dir is None:
        raise FileNotFoundError("Attach the Name That LLM competition data to this notebook.")
    output_dir = Path("/kaggle/working")
else:
    data_dir = Path(r"E:\kaggle\tfidf-baseline\data")
    output_dir = data_dir.parent

predictions = run(data_dir, output_dir / "submission.parquet")
print(predictions.head().to_string(index=False))